In [ ]:
import pandas as pd
from rouge_score import rouge_scorer, scoring

# === CONFIG ===
INPUT_FILE = "./data/LLaVA-Med/output_all_adr_subset_0811_1.csv"  # change to your CSV filename
OUTPUT_FILE = "./data/LLaVA-Med/LLAMA_NER_ROUGE_Output_1.txt"

# Initialize ROUGE scorer and aggregators
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
drug_agg = scoring.BootstrapAggregator()
adverse_agg = scoring.BootstrapAggregator()

# Read CSV file
df = pd.read_csv(INPUT_FILE)

# Ensure required columns exist
required_cols = ['Drug Name', 'drug_names', 'Side/Harmful effects', 'adverse_effects']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Prepare output
output_lines = []

# --- Process each row ---
for idx, row in df.iterrows():
    output_lines.append("----")
    output_lines.append(f"Row {idx + 1}:")

    # 1️⃣ Drug Name comparison
    ref_drug = str(row['Drug Name']).strip()
    gen_drug = str(row['drug_names']).strip()
    drug_scores = scorer.score(ref_drug, gen_drug)
    drug_agg.add_scores(drug_scores)

    output_lines.append("Drug Name:")
    output_lines.append(f"\tReference: {ref_drug}")
    output_lines.append(f"\tGenerated: {gen_drug}")
    output_lines.append("\tROUGE Score:")
    for key, val in drug_scores.items():
        output_lines.append(f"\t\t{key.upper():7} | P={val.precision:.3f}  R={val.recall:.3f}  F1={val.fmeasure:.3f}")

    # 2️⃣ Adverse Effect comparison
    ref_adv = str(row['Side/Harmful effects']).strip()
    gen_adv = str(row['adverse_effects']).strip()
    adv_scores = scorer.score(ref_adv, gen_adv)
    adverse_agg.add_scores(adv_scores)

    output_lines.append("")
    output_lines.append("Adverse Effect:")
    output_lines.append(f"\tReference: {ref_adv}")
    output_lines.append(f"\tGenerated: {gen_adv}")
    output_lines.append("\tROUGE Score:")
    for key, val in adv_scores.items():
        output_lines.append(f"\t\t{key.upper():7} | P={val.precision:.3f}  R={val.recall:.3f}  F1={val.fmeasure:.3f}")

# --- Aggregated Results ---
output_lines.append("----")
output_lines.append("Aggregated:")

# Aggregated scores
agg_drug = drug_agg.aggregate()
agg_adv = adverse_agg.aggregate()

# Function to format aggregated scores
def format_aggregate(title, agg_scores):
    lines = [f"{title}:", "\tROUGE Score:"]
    for key, val in agg_scores.items():
        lines.append(f"\t\t{key.upper():7} | P={val.mid.precision:.3f}  R={val.mid.recall:.3f}  F1={val.mid.fmeasure:.3f}")
    return lines

# Add aggregated Drug Name + Adverse Effect
output_lines += format_aggregate("Drug Name", agg_drug)
output_lines.append("")
output_lines += format_aggregate("Adverse Effect", agg_adv)

# --- Overall average (mean of both categories) ---
overall_avg = {}
for key in agg_drug.keys():
    overall_avg[key] = (
        (agg_drug[key].mid.precision + agg_adv[key].mid.precision) / 2,
        (agg_drug[key].mid.recall + agg_adv[key].mid.recall) / 2,
        (agg_drug[key].mid.fmeasure + agg_adv[key].mid.fmeasure) / 2,
    )

output_lines.append("")
output_lines.append("Overall:")
output_lines.append("\tROUGE Score:")
for key, (p, r, f1) in overall_avg.items():
    output_lines.append(f"\t\t{key.upper():7} | P={p:.3f}  R={r:.3f}  F1={f1:.3f}")

# --- Final Average F1 ---
final_f1 = sum(f1 for _, _, f1 in overall_avg.values()) / len(overall_avg)
output_lines.append("")
output_lines.append(f"----\nFinal Average F1 (ROUGE-1, ROUGE-2, ROUGE-L): {final_f1:.3f}")

# --- Write to file ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"ROUGE scores written to {OUTPUT_FILE}")
